# RG3 gas: train-normal rank fusion of RF and positive-mode experts
Original 200-tree RF preserved, two positive-mode 100-tree experts per view. Two feature views kept.
Ranks learned from train-only inner held-out normal scores, never outer test ranks. Eight fixed score functions and train-only selector.
All observations in outer query block excluded from every fit. Internal calibration may be optimistic; repeated-data best results require further validation.
Original raw baseline predictions verified identical. Current score values are ranking scores, not calibrated probabilities.

In [ ]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
setup=''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0].replace('n_jobs=-1','n_jobs=2')
exec(setup)
expert_cache={}
def expert(ref,q):
    key=(tuple(ref),tuple(q))
    if key in expert_cache:return expert_cache[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    yy=y[ref];pos=yy==1;normal=yy==0
    if len(np.unique(yy))<2:return np.zeros(len(q))
    rr=np.full((len(ref),4),np.nan)
    for b in groups.iloc[ref].unique():
        mask=groups.iloc[ref].eq(b).to_numpy()
        if (~mask).any() and (y[ref[~mask]]==0).any():rr[mask]=res_features(ref[~mask],ref[mask])
    views=[(d.iloc[ref][stdcols].to_numpy(float),d.iloc[q][stdcols].to_numpy(float)),(rr,res_features(ref,q))]
    scores=[]
    for tr,te in views:
        imp=SimpleImputer(strategy='median',keep_empty_features=True)
        tr=imp.fit_transform(tr);te=imp.transform(te)
        sc=RobustScaler().fit(tr[normal]);z=sc.transform(tr)
        k=min(2,len(np.unique(z[pos],axis=0)))
        labels=KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(z[pos]) if k>1 else np.zeros(pos.sum(),int)
        ids=np.flatnonzero(pos);out=[]
        for mode in np.unique(labels):
            chosen=np.r_[np.flatnonzero(normal),ids[labels==mode]]
            m=RandomForestClassifier(n_estimators=100,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=2)
            out.append(m.fit(tr[chosen],yy[chosen]).predict_proba(te)[:,1])
        scores.append(np.mean(out,axis=0))
    expert_cache[key]=np.max(scores,axis=0)
    return expert_cache[key]
cache2={}
def pair(ref,q):
    key=(tuple(ref),tuple(q))
    if key not in cache2:cache2[key]=np.column_stack([pair_predict(ref,q).max(axis=1),expert(ref,q)])
    return cache2[key]
def ranks(scores,normal):
    # Smoothed train-normal empirical CDF, not test-block ranks or test labels.
    return np.column_stack([(np.searchsorted(np.sort(normal[:,k]),scores[:,k],side='right')+.5)/(len(normal)+1.) for k in range(2)])
def fuse(u):
    a=u[:,0];b=u[:,1]
    return {'baseline_rank':a,'expert_rank':b,'mean25':.75*a+.25*b,'mean50':.5*a+.5*b,'mean75':.25*a+.75*b,'geometric':np.sqrt(a*b),'rank_min':np.minimum(a,b),'rank_max':np.maximum(a,b)}
records=[];choices=[];calrecords=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b))
        ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=pair(ref,q);inner=np.full((len(ref),2),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():inner[mask]=pair(ref[~mask],ref[mask])
        valid=np.isfinite(inner).all(axis=1);normal=inner[valid&(y[ref]==0)]
        cal=fuse(ranks(inner[valid],normal));out=fuse(ranks(outer,normal))
        candidates=[]
        for name,v in cal.items():
            cut=float(np.quantile(v[y[ref][valid]==0],.99,method='higher'))
            pr=v>cut;yy=y[ref][valid]
            tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
            f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
            candidates.append(dict(model=name,threshold=cut,inner_F1=f1,inner_TP=int(tp),inner_FP=int(fp)))
            for k,i in enumerate(q):records.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(out[name][k]),raw_baseline_score=float(outer[k,0]),raw_expert_score=float(outer[k,1]),threshold=cut,prediction=int(out[name][k]>cut)))
        selected=max(candidates,key=lambda c:(c['inner_F1'],-c['inner_FP'],c['model']=='baseline_rank'))
        for k,i in enumerate(q):records.append(dict(scheme=scheme,model='inner_selected',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(out[selected['model']][k]),raw_baseline_score=float(outer[k,0]),raw_expert_score=float(outer[k,1]),threshold=selected['threshold'],prediction=int(out[selected['model']][k]>selected['threshold'])))
        choices.append(dict(scheme=scheme,block=b,**selected))
        for k,i in enumerate(ref):
            calrecords.append(dict(scheme=scheme,heldout_block=b,TimeStamp=d.TimeStamp.iloc[i],block=groups.iloc[i],actual=int(y[i]),raw_baseline_score=float(inner[k,0]),raw_expert_score=float(inner[k,1])))
        print('Finished rank fusion',scheme,b,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,name),g in pp.groupby(['scheme','model'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,model=name,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,AP=average_precision_score(g.actual,g.score)))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_rank_fusion_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_rank_fusion_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(tables/'rg3_rank_fusion_inner_choices.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(calrecords).to_csv(tables/'rg3_rank_fusion_inner_scores.csv',index=False,encoding='utf-8-sig')
old=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
chk=pp.loc[pp.model.eq('baseline_rank')].merge(old.loc[old.model.eq('max_union'),['scheme','TimeStamp','score']],on=['scheme','TimeStamp'],suffixes=('_rank','_old'),validate='one_to_one')
assert len(chk)==1023 and np.allclose(chk.raw_baseline_score,chk.score_old)
print(metrics.to_string(index=False))


## Executed results
Whole fixed rank_max: TP5 FP11 FN12 TN555 F1=0.303; forward TP3 FP6 F1=0.273. Inner-selected whole TP4 FP11 F1=0.250, forward TP2 FP6 F1=0.190. Balance target not achieved. Baseline raw scores1023 verified.
See reports/rg3_iterative_improvement_log.md for protocol, limitations and source tables.